# Flat vs Sloping Seafloor Comparison

This notebook compares the baseline flat `100 m` cavity against a **near-pinch-out** cavity in which the seafloor ramps upward over the final two kilometers before the grounding line and leaves `30 m` of water at `x = 0`. The goal is to see how much this geometry change alters the water-column reverberations and the surface wavefield without changing the source, ice properties, or substrate properties.


In [ ]:
from csv import DictReader
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
results_dir = ROOT / 'results_seafloorshape_nz1_10hz'
summary_path = results_dir / 'summary.csv'
rows = list(DictReader(summary_path.open()))
rows_by_id = {row['case_id']: row for row in rows}
rows


In [ ]:
def load_case(case_id):
    data = np.load(results_dir / case_id / 'surface_gather.npz', allow_pickle=True)
    return {key: data[key] for key in data.files}

flat = load_case('flat_100m_nz1')
slope = load_case('slope_30m_nz1_dt02')

flat_row = rows_by_id['flat_100m_nz1']
slope_row = rows_by_id['slope_30m_nz1_dt02']

print('Flat cavity reflection coefficient:', float(flat_row['reflection_coefficient']))
print('Sloping cavity reflection coefficient:', float(slope_row['reflection_coefficient']))
print('Flat cavity reflected center (s):', float(flat_row['reflected_center_s']))
print('Sloping cavity reflected center (s):', float(slope_row['reflected_center_s']))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.linspace(-12000.0, 6000.0, 600)
ice_base = np.full_like(x, 1200.0)
surface = np.full_like(x, 1500.0)
flat_seafloor = np.full_like(x, 1100.0)
slope_seafloor = np.full_like(x, 1170.0)
slope_seafloor[x <= -2000.0] = 1100.0
mask = (x > -2000.0) & (x < 0.0)
slope_seafloor[mask] = 1100.0 + 70.0 * (x[mask] + 2000.0) / 2000.0

ax.plot(x, surface, color='black', lw=1.5, label='free surface')
ax.plot(x, ice_base, color='tab:cyan', lw=2, label='ice base')
ax.plot(x, flat_seafloor, color='tab:blue', lw=2, label='flat seafloor')
ax.plot(x, slope_seafloor, color='tab:orange', lw=2, label='sloping seafloor')
ax.axvline(0.0, color='tab:red', lw=1.2, ls='--', label='grounding line')
ax.set_xlim(-2500, 1000)
ax.set_ylim(1050, 1510)
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
ax.set_title('Geometry near the grounding line')
ax.grid(True, alpha=0.25)
ax.legend(ncol=2)
fig.tight_layout()


In [ ]:
def clip_for_plot(gather, percentile=99.5):
    clip = np.percentile(np.abs(gather['bxx']), percentile)
    return max(clip, 1.0e-12)

def resample_gather_to_time(gather, target_time):
    out = np.empty((gather['bxx'].shape[0], target_time.size), dtype=float)
    for i in range(gather['bxx'].shape[0]):
        out[i, :] = np.interp(target_time, gather['time'], gather['bxx'][i, :])
    return out

flat_clip = clip_for_plot(flat)
slope_clip = clip_for_plot(slope)
common_clip = max(flat_clip, slope_clip)

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True, constrained_layout=True)
for ax, gather, title in [
    (axes[0], flat, 'Flat cavity'),
    (axes[1], slope, 'Sloping cavity'),
]:
    ax.imshow(
        gather['bxx'].T,
        origin='lower',
        aspect='auto',
        extent=[gather['x'].min(), gather['x'].max(), gather['time'].min(), gather['time'].max()],
        cmap='seismic',
        vmin=-common_clip,
        vmax=common_clip,
    )
    ax.axvline(0.0, color='k', lw=1.0, ls='--')
    ax.set_title(title)
    ax.set_xlabel('x (m)')

slope_on_flat_time = resample_gather_to_time(slope, flat['time'])
difference = slope_on_flat_time - flat['bxx']
diff_clip = max(np.percentile(np.abs(difference), 99.5), 1.0e-12)
axes[2].imshow(
    difference.T,
    origin='lower',
    aspect='auto',
    extent=[flat['x'].min(), flat['x'].max(), flat['time'].min(), flat['time'].max()],
    cmap='seismic',
    vmin=-diff_clip,
    vmax=diff_clip,
)
axes[2].axvline(0.0, color='k', lw=1.0, ls='--')
axes[2].set_title('Sloping - Flat')
axes[2].set_xlabel('x (m)')
axes[0].set_ylabel('time (s)')
fig.suptitle('Surface BXX gathers')


In [ ]:
def trace_at_x(gather, x_target):
    idx = int(np.argmin(np.abs(gather['x'] - x_target)))
    return gather['x'][idx], gather['bxx'][idx, :]

x_targets = [-1800.0, -1000.0, -400.0]
fig, axes = plt.subplots(len(x_targets), 1, figsize=(10, 2.8 * len(x_targets)), sharex=True, constrained_layout=True)
for ax, x_target in zip(np.atleast_1d(axes), x_targets):
    x_flat, trace_flat = trace_at_x(flat, x_target)
    x_slope, trace_slope = trace_at_x(slope, x_target)
    scale = max(np.max(np.abs(trace_flat)), np.max(np.abs(trace_slope)), 1.0e-12)
    ax.plot(flat['time'], trace_flat / scale, label=f'flat @ {x_flat:.1f} m', lw=1.2)
    ax.plot(slope['time'], trace_slope / scale, label=f'slope @ {x_slope:.1f} m', lw=1.2)
    ax.set_ylabel('norm. BXX')
    ax.set_title(f'Trace comparison near x = {x_target:.0f} m')
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper right')
axes[-1].set_xlabel('time (s)')


## What to look for

- The sloping cavity should reduce or retime some of the water-column reverberations because the cavity thickness is no longer uniform up to the grounding line.
- Any change concentrated near the reflected packet window would suggest that the cavity geometry is contributing directly to the scattered wavefield, not just the early water reverberations.
- If the `Sloping - Flat` gather shows strong coherent differences on the floating side before the main reflected arrival, that is a good sign that the cavity mode structure has changed.
